# Day 4 — K-Means Clustering

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)


## Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering import (
    career_overlap_table,
    cluster_profile_table,
    fit_kmeans,
    inertia_by_k,
    silhouette_by_k,
)
from src.preprocessing import (
    add_engineered_features,
    load_artifacts,
    load_cleaned_table,
    prepare_train_test_bundle,
    save_artifacts,
    transform_all_profiles,
)

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

df = load_cleaned_table(PROJECT_ROOT)
try:
    preprocessor = load_artifacts(PROJECT_ROOT)["preprocessor"]
except FileNotFoundError:
    bundle = prepare_train_test_bundle(df)
    save_artifacts(bundle, PROJECT_ROOT)
    preprocessor = bundle["preprocessor"]

X = transform_all_profiles(df, preprocessor)
X.shape

## 1. Elbow and silhouette

In [ ]:
k_values = list(range(1, 11))
inertia_df = inertia_by_k(X, k_values)
sil_df = silhouette_by_k(X, [k for k in k_values if k >= 2])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(inertia_df["k"], inertia_df["inertia"], marker="o")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow method")
axes[1].plot(sil_df["k"], sil_df["silhouette"], marker="o", color="darkgreen")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette")
axes[1].set_title("Silhouette by K")
plt.tight_layout()
elbow_path = FIGURES_DIR / "day04_elbow_silhouette.png"
plt.savefig(elbow_path, dpi=120, bbox_inches="tight")
plt.show()
elbow_path

In [ ]:
CHOSEN_K = 4
kmeans = fit_kmeans(X, n_clusters=CHOSEN_K)
labels = kmeans.labels_
pd.Series(labels).value_counts().sort_index()

## 2. Cluster profiles

In [ ]:
profile_df = add_engineered_features(df)
profile_cols = ["CGPA", "Technical_score", "Academic_score", "Experience_score"]
cluster_profile_table(profile_df, labels, profile_cols)

## 3. Career mix per cluster (exploratory)

In [ ]:
career_overlap_table(df, labels).iloc[:, :6]

## 4. PCA plot

In [ ]:
pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(X)
plot_df = pd.DataFrame({"pc1": xy[:, 0], "pc2": xy[:, 1], "cluster": labels})
plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x="pc1", y="pc2", hue="cluster", palette="tab10", alpha=0.5, s=25)
plt.title(f"Profiles (PCA), K={CHOSEN_K}")
plt.tight_layout()
pca_path = FIGURES_DIR / "day04_kmeans_pca.png"
plt.savefig(pca_path, dpi=120, bbox_inches="tight")
plt.show()
pca_path

In [ ]:
out = profile_df[["Student_ID", "Career"]].copy()
out["cluster"] = labels
cluster_path = METRICS_DIR / "day04_cluster_assignments.csv"
out.to_csv(cluster_path, index=False)
cluster_path